<a href="https://colab.research.google.com/github/ekBlaise/unsloth_solution_measuring_time/blob/master/unsloth_solution_measuring_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2

In [ ]:
import unsloth  # Important: import Unsloth early

import time
import torch

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
assert torch.cuda.is_available(), "Please enable GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_SIZE = 200
MAX_SEQ_LENGTH = 1024
MAX_STEPS = 30
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-4

In [ ]:
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train").select(range(DATASET_SIZE))

alpaca_prompt = """Below is an instruction.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
"""

def convert_to_text(example):
    return {
        "text": alpaca_prompt.format(
            instruction=example["instruction"],
            input=example["input"],
            output=example["output"],
        )
    }

dataset = raw_dataset.map(
    convert_to_text,
    remove_columns=raw_dataset.column_names,
)

print("Sample training text:\n")
print(dataset[0]["text"])


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()


In [ ]:
sft_config = SFTConfig(
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,

    output_dir="unsloth_out",
    report_to="none",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    # Keep same as HF code for fair classroom comparison
    fp16=False,
    bf16=False,

    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start_time = time.time()

trainer.train()

torch.cuda.synchronize()

train_time = round(time.time() - start_time, 2)
train_vram_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
train_vram_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

print("\nUNSLOTH TRAINING RESULTS")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)

In [ ]:
FastLanguageModel.for_inference(model)

instruction = "Give three tips for staying healthy."

inference_prompt = """Below is an instruction.

### Instruction:
{instruction}

### Input:


### Response:
""".format(instruction=instruction)

inputs = tokenizer(inference_prompt, return_tensors="pt").to("cuda")

# Warm-up generation
with torch.inference_mode():
    _ = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
t0 = time.time()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
t1 = time.time()

input_tokens = inputs["input_ids"].shape[-1]
generated_token_ids = output[0][input_tokens:]

generated_tokens = len(generated_token_ids)
tokens_per_sec = round(generated_tokens / (t1 - t0), 2)

answer = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

print("\nUNSLOTH FINAL RESULTS")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)
print("Generated tokens/sec:", tokens_per_sec)

print("\nMODEL ANSWER:")
print(answer)

In [ ]:
# Same base model
# Same dataset
# Same dataset size
# Same prompt format
# Same max steps
# Same batch size
# Same gradient accumulation
# Same learning rate
# Same LoRA r
# Same LoRA alpha
# Same LoRA target modules
# Same LoRA dropout
# Same max sequence length
# Same optimizer
# Same inference prompt
# Same max new tokens
# Same do_sample=False
# Same metric calculation

| Parameter                  | Meaning                                                    |        HF Result |  Unsloth Result | Better           | Simple Conclusion                                 |
| -------------------------- | ---------------------------------------------------------- | ---------------: | --------------: | ---------------- | ------------------------------------------------- |
| **Train time/sec**         | Time taken to complete fine-tuning                         |        94.46 sec |       61.27 sec | Lower is better  | **Unsloth trained ~1.54x faster**                 |
| **Peak allocated VRAM/GB** | Actual GPU memory used by tensors during training          |         1.717 GB |        0.931 GB | Lower is better  | **Unsloth used less actual GPU memory**           |
| **Peak reserved VRAM/GB**  | GPU memory reserved by PyTorch/CUDA for current/future use |         2.186 GB |        1.693 GB | Lower is better  | **Unsloth used less overall reserved GPU memory** |
| **Generated tokens/sec**   | Number of new tokens generated per second during inference | 12.61 tokens/sec | 9.65 tokens/sec | Higher is better | **HF inference was faster in this run**           |
